In [2]:
pip install -q speechbrain torchaudio tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 22.5 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


Cài Dependency

In [7]:
!mkdir -p /kaggle/working/results/voxceleb_indian
!mkdir -p /kaggle/working/results/vivos

Tạo folder kết quả

In [11]:
!cp /kaggle/input/datasets/qnnnov6/input-code/iden_split.txt /kaggle/working/results/voxceleb_indian/iden_split.txt
!cp /kaggle/input/datasets/qnnnov6/input-code/veri_test.txt /kaggle/working/results/voxceleb_indian/veri_test.txt

Tạo split

In [16]:
from pathlib import Path
import random, json

random.seed(42)

DATA_ROOT = Path("/kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian")
OUT_DIR = Path("/kaggle/working/results/voxceleb_indian")
OUT_DIR.mkdir(parents=True, exist_ok=True)

spk2files = {}
for spk_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    rels = []
    for wav in sorted(spk_dir.rglob("*.wav")):
        rels.append(str(wav.relative_to(DATA_ROOT)).replace("\\", "/"))
    if len(rels) >= 3:
        spk2files[spk_dir.name] = rels

iden_lines = []
test_by_spk = {}
counts = {"train": 0, "val": 0, "test": 0}

for spk, files in spk2files.items():
    files = files[:]
    random.shuffle(files)
    n = len(files)
    n_train = max(1, int(0.70 * n))
    n_val = max(1, int(0.15 * n))

    train_files = files[:n_train]
    val_files = files[n_train:n_train+n_val]
    test_files = files[n_train+n_val:]

    if not test_files:
        test_files = val_files[-1:]
        val_files = val_files[:-1] or train_files[-1:]

    for p in train_files:
        iden_lines.append(f"1 {p}\n")
    for p in val_files:
        iden_lines.append(f"2 {p}\n")
    for p in test_files:
        iden_lines.append(f"3 {p}\n")

    counts["train"] += len(train_files)
    counts["val"] += len(val_files)
    counts["test"] += len(test_files)
    test_by_spk[spk] = test_files

pos, neg = [], []
spks = sorted(test_by_spk)

for files in test_by_spk.values():
    for i in range(len(files)):
        for j in range(i + 1, min(i + 4, len(files))):
            pos.append((1, files[i], files[j]))

for i in range(len(spks)):
    for j in range(i + 1, len(spks)):
        a, b = test_by_spk[spks[i]], test_by_spk[spks[j]]
        if a and b:
            neg.append((0, random.choice(a), random.choice(b)))

k = min(len(pos), len(neg))
trials = random.sample(pos, k) + random.sample(neg, k)
random.shuffle(trials)

(OUT_DIR / "iden_split.txt").write_text("".join(iden_lines))
(OUT_DIR / "veri_test.txt").write_text(
    "".join(f"{label} {p1} {p2}\n" for label, p1, p2 in trials)
)

summary = {
    "dataset": "VoxCeleb Indian subset",
    "seed": 42,
    "data_root": str(DATA_ROOT),
    "n_speakers": len(spk2files),
    "n_utterances": sum(len(v) for v in spk2files.values()),
    "split_counts": counts,
    "n_trials": len(trials),
    "n_positive_trials": sum(1 for t in trials if t[0] == 1),
    "n_negative_trials": sum(1 for t in trials if t[0] == 0),
}
(OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))

{
  "dataset": "VoxCeleb Indian subset",
  "seed": 42,
  "data_root": "/kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian",
  "n_speakers": 24,
  "n_utterances": 4857,
  "split_counts": {
    "train": 3389,
    "val": 717,
    "test": 751
  },
  "n_trials": 552,
  "n_positive_trials": 276,
  "n_negative_trials": 276
}


Prepare split

In [17]:
!python /kaggle/input/datasets/qnnnov6/input-code/train_ecapa.py \
  --data_root /kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian \
  --split_file /kaggle/working/results/voxceleb_indian/iden_split.txt \
  --save_dir /kaggle/working/results/voxceleb_indian \
  --epochs 15 --batch_size 64 --lr 1e-3 --num_workers 2

Device: cuda
Train: 3389 | Val: 717 | Speakers: 24
Epoch 01 | train_loss=4.8280 train_acc=0.3203 | val_loss=2.8577 val_acc=0.4937  
  ✓ Saved best (val_acc=0.4937)
Epoch 02 | train_loss=1.7541 train_acc=0.6629 | val_loss=3.3471 val_acc=0.4951  
  ✓ Saved best (val_acc=0.4951)
Epoch 03 | train_loss=1.0561 train_acc=0.7785 | val_loss=1.4756 val_acc=0.7127  
  ✓ Saved best (val_acc=0.7127)
Epoch 04 | train_loss=0.6774 train_acc=0.8419 | val_loss=1.2599 val_acc=0.7643  
  ✓ Saved best (val_acc=0.7643)
Epoch 05 | train_loss=0.3784 train_acc=0.9044 | val_loss=1.0566 val_acc=0.8131  
  ✓ Saved best (val_acc=0.8131)
Epoch 06 | train_loss=0.2802 train_acc=0.9312 | val_loss=0.7058 val_acc=0.8577  
  ✓ Saved best (val_acc=0.8577)
Epoch 07 | train_loss=0.1426 train_acc=0.9600 | val_loss=0.5625 val_acc=0.8759  
  ✓ Saved best (val_acc=0.8759)
Epoch 08 | train_loss=0.1143 train_acc=0.9660 | val_loss=0.5878 val_acc=0.8898  
  ✓ Saved best (val_acc=0.8898)
Epoch 09 | train_loss=0.0617 train_acc=0.9814

Train

In [18]:
!python /kaggle/input/datasets/qnnnov6/input-code/evaluate_sid.py \
  --ckpt /kaggle/working/results/voxceleb_indian/best_model.pt \
  --spk2idx /kaggle/working/results/voxceleb_indian/spk2idx.json \
  --data_root /kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian \
  --split_file /kaggle/working/results/voxceleb_indian/iden_split.txt \
  --out /kaggle/working/results/voxceleb_indian/sid_results.json

!python /kaggle/input/datasets/qnnnov6/input-code/evaluate_sv.py \
  --ckpt /kaggle/working/results/voxceleb_indian/best_model.pt \
  --data_root /kaggle/input/datasets/gaurav41/voxceleb1-audio-wav-files-for-india-celebrity/vox1_indian/content/vox_indian \
  --trial_file /kaggle/working/results/voxceleb_indian/veri_test.txt \
  --out /kaggle/working/results/voxceleb_indian/sv_results.json

Loaded checkpoint từ epoch 15
Test samples: 751
Test: 100%|█████████████████████████████████████| 12/12 [00:07<00:00,  1.63it/s]

Speaker Identification Results
  Test samples:    751
  Top-1 accuracy:  91.34%
  Top-5 accuracy:  98.67%
Loaded checkpoint từ epoch 15 (val_acc=0.9344)
Trials: 552 | Unique files: 572
Score trials: 100%|███████████████████████| 552/552 [00:00<00:00, 493763.23it/s]

Speaker Verification Results
  EER:                2.90%
  Decision threshold: 0.3884
  minDCF (p=0.01):    0.1486
Đã lưu kết quả vào /kaggle/working/results/voxceleb_indian/sv_results.json


Evaluate

In [24]:
from pathlib import Path
import random, json
import torchaudio

random.seed(42)

DATA_ROOT = Path("/kaggle/working/vivos_speaker_root")
OUT_DIR = Path("/kaggle/working/results/vivos")
OUT_DIR.mkdir(parents=True, exist_ok=True)

bad = []
spk2files = {}

for spk_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    good = []
    for wav in sorted(spk_dir.glob("*.wav")):
        rel = f"{spk_dir.name}/{wav.name}"
        try:
            torchaudio.load(str(wav))
            good.append(rel)
        except Exception as e:
            bad.append((rel, str(e).splitlines()[0]))
    if len(good) >= 3:
        spk2files[spk_dir.name] = good

iden_lines = []
test_by_spk = {}
counts = {"train": 0, "val": 0, "test": 0}

for spk, files in spk2files.items():
    files = files[:]
    random.shuffle(files)

    n = len(files)
    n_train = max(1, int(0.70 * n))
    n_val = max(1, int(0.15 * n))

    train_files = files[:n_train]
    val_files = files[n_train:n_train+n_val]
    test_files = files[n_train+n_val:]

    if not test_files:
        test_files = val_files[-1:]
        val_files = val_files[:-1] or train_files[-1:]

    for p in train_files:
        iden_lines.append(f"1 {p}\n")
    for p in val_files:
        iden_lines.append(f"2 {p}\n")
    for p in test_files:
        iden_lines.append(f"3 {p}\n")

    counts["train"] += len(train_files)
    counts["val"] += len(val_files)
    counts["test"] += len(test_files)
    test_by_spk[spk] = test_files

pos, neg = [], []
spks = sorted(test_by_spk)

for files in test_by_spk.values():
    for i in range(len(files)):
        for j in range(i + 1, min(i + 4, len(files))):
            pos.append((1, files[i], files[j]))

for i in range(len(spks)):
    for j in range(i + 1, len(spks)):
        a, b = test_by_spk[spks[i]], test_by_spk[spks[j]]
        if a and b:
            neg.append((0, random.choice(a), random.choice(b)))

k = min(len(pos), len(neg))
trials = random.sample(pos, k) + random.sample(neg, k)
random.shuffle(trials)

(OUT_DIR / "iden_split.txt").write_text("".join(iden_lines))
(OUT_DIR / "veri_test.txt").write_text(
    "".join(f"{label} {p1} {p2}\n" for label, p1, p2 in trials)
)
(OUT_DIR / "bad_audio_files.txt").write_text(
    "\n".join(f"{rel}\t{err}" for rel, err in bad)
)

summary = {
    "dataset": "VIVOS",
    "seed": 42,
    "data_root": str(DATA_ROOT),
    "n_speakers": len(spk2files),
    "n_utterances": sum(len(v) for v in spk2files.values()),
    "split_counts": counts,
    "n_trials": len(trials),
    "n_positive_trials": sum(1 for t in trials if t[0] == 1),
    "n_negative_trials": sum(1 for t in trials if t[0] == 0),
    "n_bad_audio_files": len(bad),
    "bad_audio_out": str(OUT_DIR / "bad_audio_files.txt"),
}
(OUT_DIR / "vivos_summary.json").write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("Bad examples:", bad[:5])

{
  "dataset": "VIVOS",
  "seed": 42,
  "data_root": "/kaggle/working/vivos_speaker_root",
  "n_speakers": 65,
  "n_utterances": 12419,
  "split_counts": {
    "train": 8686,
    "val": 1850,
    "test": 1883
  },
  "n_trials": 4160,
  "n_positive_trials": 2080,
  "n_negative_trials": 2080,
  "n_bad_audio_files": 1,
  "bad_audio_out": "/kaggle/working/results/vivos/bad_audio_files.txt"
}
Bad examples: [('VIVOSSPK44/train_VIVOSSPK44_157.wav', 'Failed to create AudioDecoder for /kaggle/working/vivos_speaker_root/VIVOSSPK44/train_VIVOSSPK44_157.wav: Could not open input file: /kaggle/working/vivos_speaker_root/VIVOSSPK44/train_VIVOSSPK44_157.wav Invalid data found when processing input')]


Prepare

In [25]:
!python /kaggle/input/datasets/qnnnov6/input-code/train_ecapa.py \
  --data_root /kaggle/working/vivos_speaker_root \
  --split_file /kaggle/working/results/vivos/iden_split.txt \
  --save_dir /kaggle/working/results/vivos \
  --epochs 15 --batch_size 64 --lr 1e-3 --num_workers 2

Device: cuda
Train: 8686 | Val: 1850 | Speakers: 65
Epoch 01 | train_loss=2.4511 train_acc=0.6019 | val_loss=1.0959 val_acc=0.7389  
  ✓ Saved best (val_acc=0.7389)
Epoch 02 | train_loss=0.4320 train_acc=0.8907 | val_loss=1.2096 val_acc=0.7459  
  ✓ Saved best (val_acc=0.7459)
Epoch 03 | train_loss=0.2003 train_acc=0.9494 | val_loss=0.5840 val_acc=0.8557  
  ✓ Saved best (val_acc=0.8557)
Epoch 04 | train_loss=0.1005 train_acc=0.9738 | val_loss=0.2355 val_acc=0.9432  
  ✓ Saved best (val_acc=0.9432)
Epoch 05 | train_loss=0.0492 train_acc=0.9868 | val_loss=0.1051 val_acc=0.9751  
  ✓ Saved best (val_acc=0.9751)
Epoch 06 | train_loss=0.0238 train_acc=0.9948 | val_loss=0.0754 val_acc=0.9778  
  ✓ Saved best (val_acc=0.9778)
Epoch 07 | train_loss=0.0101 train_acc=0.9978 | val_loss=0.0793 val_acc=0.9838  
  ✓ Saved best (val_acc=0.9838)
Epoch 08 | train_loss=0.0147 train_acc=0.9978 | val_loss=0.0459 val_acc=0.9876  
  ✓ Saved best (val_acc=0.9876)
Epoch 09 | train_loss=0.0078 train_acc=0.999

Train

In [26]:
!python /kaggle/input/datasets/qnnnov6/input-code/evaluate_sid.py \
  --ckpt /kaggle/working/results/vivos/best_model.pt \
  --spk2idx /kaggle/working/results/vivos/spk2idx.json \
  --data_root /kaggle/working/vivos_speaker_root \
  --split_file /kaggle/working/results/vivos/iden_split.txt \
  --out /kaggle/working/results/vivos/sid_results.json

!python /kaggle/input/datasets/qnnnov6/input-code/evaluate_sv.py \
  --ckpt /kaggle/working/results/vivos/best_model.pt \
  --data_root /kaggle/working/vivos_speaker_root \
  --trial_file /kaggle/working/results/vivos/veri_test.txt \
  --out /kaggle/working/results/vivos/sv_results.json

Loaded checkpoint từ epoch 10
Test samples: 1883
Test: 100%|█████████████████████████████████████| 30/30 [00:11<00:00,  2.61it/s]

Speaker Identification Results
  Test samples:    1883
  Top-1 accuracy:  99.31%
  Top-5 accuracy:  100.00%
Loaded checkpoint từ epoch 10 (val_acc=0.9919)
Trials: 4160 | Unique files: 1858
Score trials: 100%|█████████████████████| 4160/4160 [00:00<00:00, 497939.69it/s]

Speaker Verification Results
  EER:                0.96%
  Decision threshold: 0.4062
  minDCF (p=0.01):    0.0673
Đã lưu kết quả vào /kaggle/working/results/vivos/sv_results.json


Evaluate

In [27]:
!python /kaggle/input/datasets/qnnnov6/additional-files/build_training_comparison_report.py \
  --voxceleb-dir /kaggle/working/results/voxceleb_indian \
  --vivos-dir /kaggle/working/results/vivos \
  --out /kaggle/working/results/training_dataset_comparison.md

Wrote /kaggle/working/results/training_dataset_comparison.md


Sinh report

In [29]:
import shutil
shutil.make_archive("/kaggle/working/secva_training_evidence", "zip", "/kaggle/working/results")
print("/kaggle/working/secva_training_evidence.zip")

/kaggle/working/secva_training_evidence.zip


Zip và tải về

In [30]:
from IPython.display import FileLink
FileLink(r'secva_training_evidence.zip')

/kaggle/working/secva_training_evidence.zip